<a href="https://colab.research.google.com/github/tallclub/matimo/blob/main/docs/notebooks/AgentWithMatimo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Autonomous Agents with Matimo

> **Matimo** is a configuration-driven SDK for building autonomous AI agents with 100+ built-in tools and a governance engine.
> 
> This notebook demonstrates how to use Matimo with LangChain and CrewAI to build intelligent agents.

**What You'll Learn:**
- Initialize Matimo and explore available tools
- Convert Matimo tools for LangChain and CrewAI
- Build autonomous agents that use Matimo tools  
- Understand Matimo's governance and policy engine

**Requirements:**
- `OPENAI_API_KEY` for LLM integration (required)
- No other API keys needed — all tools are built-in

## Setup: Install Dependencies

> **Note for Colab Users:** This notebook works perfectly in Google Colab!

In [1]:
# Install latest versions of all dependencies
!pip install --upgrade matimo langchain langchain-core langchain-openai crewai -q
print('✅ All dependencies installed (latest versions)')



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
✅ All dependencies installed (latest versions)


In [2]:
# Verify all packages are installed correctly
print('📦 Checking package versions...\n')

from importlib.metadata import version

packages = [
    'matimo',
    'langchain',
    'langchain-core',
    'langchain-openai',
    'crewai',
]

for package_name in packages:
    try:
        pkg_version = version(package_name)
        display_name = package_name.replace('-', '_')
        print(f'✅ {display_name:<20} v{pkg_version}')
    except Exception as e:
        print(f'⚠️ {package_name:<20} (install check: {str(e)[:30]})')

print('\n✅ All packages ready!')

📦 Checking package versions...

✅ matimo               v0.1.1.post1
✅ langchain            v1.3.0
✅ langchain_core       v1.4.0
✅ langchain_openai     v1.2.1
✅ crewai               v1.14.4

✅ All packages ready!


In [3]:
# Import required modules
import os
import asyncio
import tempfile
import json
import getpass
from pathlib import Path

# Import from matimo - Works in Colab and after pip install matimo
# (After meta-package fix, this will work everywhere)
from matimo import Matimo
from matimo import convert_tools_to_langchain, convert_tools_to_crewai

# LangChain
from langchain_openai import ChatOpenAI

# CrewAI imports
from crewai import Agent, Task, Crew

print('✅ All imports successful')

✅ All imports successful


In [4]:
# Configure OpenAI API key
if not os.environ.get('OPENAI_API_KEY'):
    api_key = getpass.getpass('Enter your OpenAI API Key: ')
    os.environ['OPENAI_API_KEY'] = api_key
    print('✅ OpenAI API key configured')
else:
    print('✅ OpenAI API key already configured')

# Create temp directory for tool storage
tools_dir = tempfile.mkdtemp(prefix='matimo_tools_')
Path(tools_dir).mkdir(parents=True, exist_ok=True)
print(f'✅ Tools directory: {tools_dir}')

✅ OpenAI API key configured
✅ Tools directory: /var/folders/1d/5sj004_10236f7xwyyjy5z5w0000gn/T/matimo_tools_gb0j_b34


In [5]:
# Initialize LLM — done here so it's always available before any agent cell
from langchain_openai import ChatOpenAI

print('Setting up LLM...')
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
print('✅ LLM ready (gpt-4o-mini)')


Setting up LLM...
✅ LLM ready (gpt-4o-mini)


In [16]:
# Workaround: agent frameworks (LangChain, CrewAI) include every schema parameter in the
# params dict and set unspecified optional ones to None.  The installed Matimo release
# converts optional int params with int(params.get("limit", 20)) which crashes on None.
# This patch strips None values before the function tool receives them — no release needed.
# The underlying fix is already merged in source and will ship in the next release.
#
# Re-run-safe: stores the true original on the class so double-patching is harmless.
from matimo.executors.function_executor import FunctionExecutor as _FE

# Save the true original once; subsequent re-runs reuse it instead of chaining patches
if not hasattr(_FE, '_matimo_orig_execute'):
    _FE._matimo_orig_execute = _FE.execute

async def _sanitized_execute(self, tool, params, credentials=None, **kwargs):  # type: ignore[no-untyped-def]
    clean = {k: v for k, v in params.items() if v is not None}
    return await _FE._matimo_orig_execute(self, tool, clean, credentials, **kwargs)

_FE.execute = _sanitized_execute  # affects all existing & future instances
print('✅ Param sanitizer applied (workaround: strips None optional params)')
print('   Affects all agent frameworks — LangChain, CrewAI, direct execute()')


✅ Param sanitizer applied (workaround: strips None optional params)
   Affects all agent frameworks — LangChain, CrewAI, direct execute()


---

# Part 1: LangChain ReAct Agent with Matimo Tools

An agent with a plain LLM can reason — but it can't act.  
Add Matimo tools and the same agent can **discover**, **validate**, and **govern** any tool in the registry.

This section shows a LangChain ReAct agent that uses Matimo's **meta-tools** to explore the registry:

| Step | Agent action | Matimo meta-tool |
|------|-------------|-----------------|
| 1 | Find all Slack integrations | `matimo_search_tools(query="slack")` |
| 2 | Find GitHub integrations | `matimo_search_tools(query="github")` |
| 3 | Find governance utilities | `matimo_search_tools(query="validate")` |
| 4 | Summarise capabilities | LLM reasoning over results |

## Initialize Matimo and Set Up LangChain Agent


In [17]:
# Initialize Matimo
print('🚀 Initializing Matimo...')

from matimo.decorators import set_global_matimo_instance

matimo = await Matimo.init(auto_discover=True)

# Register as global instance so meta-tools (matimo_search_tools, matimo_validate_tool, etc.)
# can find the loaded registry when called by agents
set_global_matimo_instance(matimo)

# List available tools
all_tools = matimo.list_tools()
print(f'✅ Matimo initialized with {len(all_tools)} tools')
print(f'✅ Global instance registered — meta-tools will use this registry')

# Show meta-tools
meta_tools = [t for t in all_tools if t.name.startswith('matimo_')]
print(f'\n📦 Meta-Tools available: {len(meta_tools)}')
for tool in sorted(meta_tools, key=lambda t: t.name)[:5]:
    print(f'   • {tool.name}')


2026-05-13T21:33:39 [matimo] INFO Matimo initialised — 18 tool(s) loaded from 1 path(s)


🚀 Initializing Matimo...
✅ Matimo initialized with 18 tools
✅ Global instance registered — meta-tools will use this registry

📦 Meta-Tools available: 12
   • matimo_approve_tool
   • matimo_create_skill
   • matimo_create_tool
   • matimo_get_skill
   • matimo_get_tool


In [18]:
# Convert Matimo tools to LangChain format (latest pattern)
print('Converting Matimo tools to LangChain format...')

# Take first 50 tools for LLM context window
langchain_tools = convert_tools_to_langchain(all_tools[:50], matimo)
print(f'✅ Converted {len(langchain_tools)} tools to LangChain format')


Converting Matimo tools to LangChain format...
✅ Converted 18 tools to LangChain format


## Demo: Exploring Tools and Capabilities

In [19]:
# Show available tools
print('📚 Sample Tools Available:\n')
for i, tool in enumerate(sorted(all_tools[:20], key=lambda t: t.name)):
    print(f'{i+1:2}. {tool.name:<25} {tool.description[:45]}')
    
print(f'\n... and {len(all_tools) - 20} more tools')
print(f'\n✅ Total: {len(all_tools)} tools ready to use')

📚 Sample Tools Available:

 1. calculator                Perform basic arithmetic operations
 2. edit                      Edit file contents with precise line-based in
 3. execute                   Execute shell commands and capture output. Su
 4. matimo_approve_tool       Approve a draft tool for production use. Re-v
 5. matimo_create_skill       Create a new skill following the Agent Skills
 6. matimo_create_tool        Create a new tool definition on disk. Validat
 7. matimo_get_skill          Level 2 activation / Level 3 resource access 
 8. matimo_get_tool           Retrieve the full definition of a tool — raw 
 9. matimo_get_tool_status    Get the current status, risk level, and appro
10. matimo_list_skills        Level 1 metadata discovery — list all skills 
11. matimo_list_user_tools    List all user-created tools in a directory wi
12. matimo_reload_tools       Hot-reload all tools from configured toolPath
13. matimo_search_tools       Search the loaded tool registry by keywor

In [20]:
# Demonstrate direct tool execution
print('🔧 Direct Tool Execution Example:\n')

# Get a simple tool (calculator)
calc_tool = matimo.get_tool('calculator')
if calc_tool:
    print(f'Tool: {calc_tool.name}')
    print(f'Description: {calc_tool.description}')
    print(f'Parameters: {list(calc_tool.parameters.keys())}')
    print(f'\n✅ Tool ready to be called by agents')
else:
    print('✅ Tools are ready for agent integration')

🔧 Direct Tool Execution Example:

Tool: calculator
Description: Perform basic arithmetic operations
Parameters: ['operation', 'a', 'b']

✅ Tool ready to be called by agents


## Part 1: LangChain Agent — Focused Discovery

Each agent gets **only the tools it needs** for one specific task.

| Cell | Agent tools | Task |
|------|-------------|------|
| Next ↓ | `matimo_search_tools` only | Ask "What Slack tools exist?" — one call, stop |
| After ↓ | *(no agent)* | Direct `matimo.execute()` for all 3 searches |

> **Why split?** Fewer tools = fewer wrong decisions. `recursion_limit=5` prevents infinite loops.


In [21]:
# Convert tools to CrewAI format
print('Converting tools to CrewAI format...')
crewai_tools = convert_tools_to_crewai(all_tools[:30], matimo)
print(f'✅ Converted {len(crewai_tools)} tools for CrewAI')

Converting tools to CrewAI format...
✅ Converted 18 tools for CrewAI


In [22]:
# Part 1 — LangChain agent with ONE tool and ONE explicit task
# Pattern from 03_meta_tools: laser-focused agent, minimal tools, explicit stop instruction
import time
from langchain.agents import create_agent

print('='*60)
print('PART 1: LangChain agent — "What meta tools does Matimo have?"')
print('='*60)

# Give the agent ONLY matimo_search_tools — nothing else to wander to
search_lc = [t for t in langchain_tools if t.name == 'matimo_search_tools']
print(f'\nAgent tools: {[t.name for t in search_lc]}  |  recursion_limit=5')
print('System prompt: call once with query="matimo_", report names, stop.\n')

agent_discover = create_agent(
    model=llm,
    tools=search_lc,
    system_prompt=(
        'You are a capability-discovery agent. '
        'Call matimo_search_tools exactly once with query="matimo_". '
        'Report the tool names you find. '
        'Stop immediately after the call — do not call any other tool.'
    ),
)

start = time.time()
try:
    result_discover = await agent_discover.ainvoke({
        'messages': [{
            'role': 'user',
            'content': 'What meta tools does Matimo have? Call matimo_search_tools with query="matimo_".'
        }]
    }, config={'recursion_limit': 5})
    elapsed = time.time() - start

    tools_called = [
        call['name']
        for msg in result_discover.get('messages', [])
        if hasattr(msg, 'tool_calls') and msg.tool_calls
        for call in msg.tool_calls
    ]
    print(f'✓ Finished in {elapsed:.1f}s  |  Tools called: {tools_called}')

    final = result_discover['messages'][-1]
    if hasattr(final, 'content'):
        print(f'\n📝 Agent answer:\n{final.content[:400]}')
except Exception as e:
    print(f'⚠ Agent error: {type(e).__name__}: {e}')


PART 1: LangChain agent — "What meta tools does Matimo have?"

Agent tools: ['matimo_search_tools']  |  recursion_limit=5
System prompt: call once with query="matimo_", report names, stop.

✓ Finished in 13.4s  |  Tools called: ['matimo_search_tools']

📝 Agent answer:
The following meta tools were found in the Matimo registry:

1. **matimo_approve_tool**
   - Description: Approve a draft tool for production use. Re-validates the tool, requires admin role, signs with HMAC, and updates the approval manifest.
   - Tags: matimo, meta, approval

2. **matimo_validate_skill**
   - Description: Validate an existing skill against the Agent Skills specification. Checks S


In [23]:
# Direct registry search — no agent needed, same matimo.execute() API
# Shows all three categories: integrations, governance, validation
print('Direct search (no LLM cost, same results):')
print()

for query in ['slack', 'github', 'validate']:
    result = await matimo.execute('matimo_search_tools', {'query': query})
    names = [r['name'] for r in result.get('results', [])]
    print(f'  query="{query}"  →  {len(names)} tools: {names[:4]}{"..." if len(names) > 4 else ""}')

print()
print('✅ Agent + direct execution both use the same Matimo registry.')
print('   Use agents when you need LLM reasoning; use matimo.execute() when you just need data.')


Direct search (no LLM cost, same results):

  query="slack"  →  0 tools: []
  query="github"  →  0 tools: []
  query="validate"  →  6 tools: ['matimo_approve_tool', 'matimo_validate_skill', 'matimo_create_tool', 'matimo_validate_tool']...

✅ Agent + direct execution both use the same Matimo registry.
   Use agents when you need LLM reasoning; use matimo.execute() when you just need data.


## Part 2: CrewAI Governance Pipeline — Two Independent Agents

Same principle: each agent has one job and only the tools for that job.

| Cell | Agent | Tools | Task |
|------|-------|-------|------|
| Next ↓ | Governance Validator | `matimo_validate_tool` only | Validate the YAML, return risk + errors |
| After ↓ | Deployment Advisor | *(none)* | Read validation output, write deployment advice |

The Advisor reads `validation_result` from the Validator's cell — no shared tool context needed.  
This mirrors the `validate → approve → execute` lifecycle that Matimo enforces on every tool.


In [24]:
# Part 2 — CrewAI, Phase 1: Governance Validator
# Agent gets ONLY matimo_validate_tool — one job, explicit stop instruction
from crewai import Agent, Task, Crew

print('='*60)
print('PART 2: CrewAI Governance Pipeline')
print('='*60)

# Only give the Validator the one tool it needs
validate_crewai = [t for t in crewai_tools if t.name == 'matimo_validate_tool']
print(f'\nValidator tools: {[t.name for t in validate_crewai]}')

SAMPLE_YAML = """name: weather_fetcher
description: Fetch current weather data for any city
version: '1.0.0'
parameters:
  city:
    type: string
    required: true
    description: City name (e.g. London)
execution:
  type: http
  method: GET
  url: 'https://wttr.in/{city}?format=j1'
"""

validator = Agent(
    role='Governance Validator',
    goal='Run matimo_validate_tool on the provided YAML. Report valid, riskLevel, schemaErrors, policyViolations.',
    backstory='A compliance engineer who validates Matimo tool definitions before deployment.',
    llm='gpt-4o-mini',   # CrewAI requires string model name — not a LangChain object
    tools=validate_crewai,
    verbose=False,
    max_iter=3,           # Never loop more than 3 times
)

validate_task = Task(
    description=(
        'Call matimo_validate_tool with the yaml_content below. '
        'Return valid (true/false), riskLevel, schemaErrors, and policyViolations.\n\n'
        f'yaml_content:\n{SAMPLE_YAML}'
    ),
    agent=validator,
    expected_output='JSON-style result: valid, riskLevel, schemaErrors list, policyViolations list.',
)

print('\nPhase 1: Validator running...\n')
try:
    crew_v = Crew(agents=[validator], tasks=[validate_task], verbose=False)
    validation_result = crew_v.kickoff()
    print('✅ Validation complete')
    print(f'\n📋 Validation output:\n{str(validation_result)[:400]}')
except Exception as e:
    print(f'⚠ Validator error: {type(e).__name__}: {e}')
    validation_result = None


PART 2: CrewAI Governance Pipeline

Validator tools: ['matimo_validate_tool']

Phase 1: Validator running...

✅ Validation complete

📋 Validation output:
{"valid":true,"riskLevel":"low","schemaErrors":[],"policyViolations":[{"rule":"forced-approval","severity":"high","message":"Untrusted tools must set requires_approval: true"},{"rule":"forced-draft-status","severity":"medium","message":"Untrusted tool has status 'stable'; must be 'draft' or unset"}]}


╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                       

In [25]:
# Part 2 — CrewAI, Phase 2: Deployment Advisor
# No tools needed — pure LLM reasoning over the Validator's output
from crewai import Agent, Task, Crew

if validation_result is None:
    print('⚠ Skipping: run the Validator cell first.')
else:
    reporter = Agent(
        role='Deployment Advisor',
        goal='Explain what the tool does, whether it passed governance, and the next deployment step.',
        backstory='A technical writer who translates validation results into clear deployment guidance.',
        llm='gpt-4o-mini',
        tools=[],       # No tools — this agent only reasons
        verbose=False,
        max_iter=2,
    )

    report_task = Task(
        description=(
            'Given this validation result:\n'
            f'{str(validation_result)[:300]}\n\n'
            'Write two sentences: '
            '(1) What the weather_fetcher tool does and whether it passed Matimo governance. '
            '(2) What the next step is before it can be executed (hint: matimo_approve_tool).'
        ),
        agent=reporter,
        expected_output='Two sentences: tool purpose + governance status, then next step.',
    )

    print('Phase 2: Deployment Advisor running...\n')
    try:
        crew_r = Crew(agents=[reporter], tasks=[report_task], verbose=False)
        report_result = crew_r.kickoff()
        print('✅ Report complete')
        print(f'\n📝 Deployment advice:\n{str(report_result)[:400]}')
    except Exception as e:
        print(f'⚠ Reporter error: {type(e).__name__}: {e}')


Phase 2: Deployment Advisor running...



✅ Report complete

📝 Deployment advice:
The weather_fetcher tool is designed to gather and provide weather information, but it did not pass Matimo governance due to high-severity policy violations related to approval requirements and tool status. The next step before it can be executed is to run the matimo_approve_tool to address the forced approval and forced draft status issues.


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [26]:
# Summary: How Autonomous Agents Become More Capable with Matimo
print('\n' + '='*60)
print('🤖 Autonomous Agents with Matimo')
print('='*60 + '\n')

print('WITHOUT MATIMO: Agent can reason but cannot act')
print('    → LLM alone has no tools, no external state\n')

print('WITH MATIMO: Agent gains capabilities\n')

print('1️⃣  DISCOVERY: Agent explores what tools exist')
print(f'    → matimo.list_tools() returns {len(all_tools)} ready-to-use tools\n')

print('2️⃣  CONVERSION: Tools adapt to any agent framework')
print(f'    → LangChain: {len(langchain_tools)} StructuredTool objects')
print(f'    → CrewAI:    {len(crewai_tools)} BaseTool objects\n')

print('3️⃣  GOVERNANCE: Every tool call passes the policy engine')
print('    → Risk classification (low → critical)')
print('    → Draft gate: validate → approve before execute\n')

print('4️⃣  AUTONOMY: Multi-step, multi-agent workflows')
print('    → matimo_search_tools  → discover integrations')
print('    → matimo_validate_tool → check YAML before deploy')
print('    → matimo_approve_tool  → promote draft to stable\n')

print('='*60)
print('✅ Your agents are ready for production use with Matimo!')
print('='*60)



🤖 Autonomous Agents with Matimo

WITHOUT MATIMO: Agent can reason but cannot act
    → LLM alone has no tools, no external state

WITH MATIMO: Agent gains capabilities

1️⃣  DISCOVERY: Agent explores what tools exist
    → matimo.list_tools() returns 18 ready-to-use tools

2️⃣  CONVERSION: Tools adapt to any agent framework
    → LangChain: 18 StructuredTool objects
    → CrewAI:    18 BaseTool objects

3️⃣  GOVERNANCE: Every tool call passes the policy engine
    → Risk classification (low → critical)
    → Draft gate: validate → approve before execute

4️⃣  AUTONOMY: Multi-step, multi-agent workflows
    → matimo_search_tools  → discover integrations
    → matimo_validate_tool → check YAML before deploy
    → matimo_approve_tool  → promote draft to stable

✅ Your agents are ready for production use with Matimo!


---
## Summary: How Agents Become More Capable with Matimo

A raw LLM can only reason. Matimo turns that reasoning into **action** by supplying a governed, registry-backed toolset that works with every major agent framework.

### Part 1: LangChain ReAct Agent
- ✅ **Tool conversion**: `convert_tools_to_langchain()` wraps all Matimo tools as LangChain `StructuredTool`
- ✅ **Agent creation**: `create_agent(model=llm, tools=lc_tools, system_prompt=...)` — LangChain 1.x API
- ✅ **Autonomous discovery**: agent calls `matimo_search_tools` three times and summarises Slack, GitHub, and governance capabilities

### Part 2: CrewAI Multi-Agent Crew
- ✅ **Tool conversion**: `convert_tools_to_crewai()` wraps Matimo tools as CrewAI `BaseTool`
- ✅ **Specialisation**: Governance Validator + Deployment Advisor with distinct roles
- ✅ **Governance pipeline**: Validator calls `matimo_validate_tool`, Advisor interprets and advises
- ✅ **Sequential context**: Reporter task receives Validator output via `context=[validate_task]`

### Key Matimo Meta-Tools
| Tool | Purpose |
|------|---------|
| `matimo_search_tools` | Search the registry by keyword — discover what's available |
| `matimo_create_tool` | Write a new YAML tool definition (`status: draft`) |
| `matimo_validate_tool` | Check YAML against schema & policy before deploying |
| `matimo_approve_tool` | Promote a draft tool to `stable` (HMAC signed) |
| `matimo_reload_tools` | Hot-reload the tool registry after changes |

### Integration Patterns

```python
# LangChain
from langchain.agents import create_agent
lc_tools = convert_tools_to_langchain(matimo.list_tools(), matimo)
agent = create_agent(model=llm, tools=lc_tools, system_prompt="...")
result = await agent.ainvoke({"messages": [{"role": "user", "content": "..."}]})

# CrewAI  (llm= must be a string model name, not a LangChain object)
from crewai import Agent, Task, Crew
crewai_tools = convert_tools_to_crewai(matimo.list_tools(), matimo)
agent = Agent(role="...", goal="...", backstory="...", llm="gpt-4o-mini", tools=crewai_tools)
task  = Task(description="...", agent=agent, expected_output="...")
crew  = Crew(agents=[agent], tasks=[task])
result = crew.kickoff()

# Direct execution (no agent LLM needed)
result = await matimo.execute("matimo_search_tools", {"query": "slack"})
result = await matimo.execute("matimo_validate_tool", {"yaml_content": yaml_str})
```

### Next Steps
- Add Slack, GitHub, or Postgres provider tools to expand agent capabilities
- Add a HITL approval callback (`on_hitl=`) to keep humans in the loop
- Explore the full governance lifecycle in **Notebook 02 — Policy Engine**
- Try the meta-tool self-creation workflow in **Notebook 03 — Meta-Tools**
